In [1]:
# Core
import os
from dotenv import load_dotenv
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go

# Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    sum as spark_sum
)

# Environment
load_dotenv()

True

# Preprocessing

## Setup

In [2]:
spark = SparkSession.builder \
    .appName("midterm") \
    .master("local[*]") \
    .getOrCreate()

25/12/14 00:51:34 WARN Utils: Your hostname, Lianas-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.20.123 instead (on interface en0)
25/12/14 00:51:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 00:51:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


These are normal on macOS and don’t affect Spark.
Macs don’t have native Hadoop libraries, so Spark uses its built-in Java version, which works fine. The hostname warning is also expected when running Spark locally.

So everything is working correctly.

In [3]:
tx_path = os.getenv("TX_PATH")
art_path = os.getenv("ART_PATH")

In [4]:
transactions = spark.read.csv(tx_path, header=True, inferSchema=True)
articles = spark.read.csv(art_path, header=True, inferSchema=True)

In [5]:
transactions.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [6]:
articles.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

In [7]:
transactions_count = transactions.count()
transactions_count

336078

In [8]:
articles_count = articles.count()
articles_count

105542

In [9]:
transactions.show(10)

+----------+--------------------+----------+------------------+----------------+
|     t_dat|         customer_id|article_id|             price|sales_channel_id|
+----------+--------------------+----------+------------------+----------------+
|2019-07-24|192e9ad7f0c05d89e...| 692721005|0.0121864406779661|               2|
|2019-04-07|70c1ee207c64a6523...| 599502013|0.0508305084745762|               2|
|2019-05-21|e29656435a0c04ef1...| 737260001|0.0254067796610169|               2|
|2019-03-29|13d1fd878959e117e...| 717251003|0.0169322033898305|               2|
|2019-07-05|58ddbcfa96eab2b3a...| 795675003| 0.008457627118644|               1|
|2019-06-04|eedec5ac3f7a88f41...| 742092001|0.0254067796610169|               2|
|2018-10-05|3de91e932fd943b2d...| 539197011|0.0135423728813559|               2|
|2019-06-04|810ed6f2725876b9d...| 372860002|0.0135423728813559|               1|
|2018-10-12|4de96a646f7030e83...| 645709001| 0.020322033898305|               2|
|2019-06-08|7d17548a65f00b2a

In [10]:
articles.show(10)

+----------+------------+--------------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|           prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|      index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+--------------------+----

In [11]:
transactions.select("customer_id").distinct().count()

230688

In [12]:
transactions.select("article_id").distinct().count()

41405

In [13]:
articles.select("article_id").distinct().count()

105542

In [14]:
transactions.describe("price").show()

+-------+-------------------+
|summary|              price|
+-------+-------------------+
|  count|             336078|
|   mean| 0.0274985190080482|
| stddev|0.01921244097171475|
|    min|  1.864406779661E-4|
|    max| 0.5067796610169492|
+-------+-------------------+



In [15]:
transactions.groupBy("sales_channel_id").count().show()

+----------------+------+
|sales_channel_id| count|
+----------------+------+
|               1|103464|
|               2|232614|
+----------------+------+



## Duplicate Check

In [16]:
transactions.count(), transactions.dropDuplicates().count()

(336078, 335151)

In [17]:
dupes = (
    transactions
    .groupBy(transactions.columns)
    .count()
    .filter("count > 1")
)

dupes.count()

895

In [18]:
dupes.show(truncate=False)

+----------+----------------------------------------------------------------+----------+------------------+----------------+-----+
|t_dat     |customer_id                                                     |article_id|price             |sales_channel_id|count|
+----------+----------------------------------------------------------------+----------+------------------+----------------+-----+
|2019-08-25|c025ac2ada70d4b94fa156ecf0ad948dc1791d74b8a194e654bd3d91a19a2a02|781683005 |0.0423559322033898|2               |2    |
|2018-12-13|97a788e36864a5634c518ad1b1ddc87da50169cd11b625f01490216f94a9042d|524825011 |0.0423559322033898|2               |2    |
|2019-05-16|e62b39733d3c123a32d62118df63317eb5b77c4b315e22151a98ce0264326bf9|722437001 |0.0220169491525423|2               |2    |
|2019-01-09|98c5c4ee85fab57b4fcf0b9b1055e8065fb454cdd92aa22929ab55e1d363475a|655248001 |0.0118474576271186|2               |2    |
|2019-04-06|7ba2cdcb79d4596615e1bbeaec8bf78328a64aaa7d166e9ec48558839ba19bdb|766346

In [19]:
transactions_clean = transactions.dropDuplicates()
transactions.count(), transactions_clean.count()

(336078, 335151)

Before analysis, I checked whether the transactions dataset contained fully identical duplicate rows.
Using transactions.count() and transactions.dropDuplicates().count(), I found:
- Original rows: 336,078
- Unique rows: 335,151
- Exact duplicates removed: 927

The duplicated rows were identical across all fields (t_dat, customer_id, article_id, price, sales_channel_id).
These duplicates are almost certainly caused by data export/ETL repetition, not by customers buying the same item twice at the same moment.

To avoid artificially inflating:
- number of transactions
- revenue
- customer-level metrics

I removed these duplicates using dropDuplicates().

## Nulls check

In [20]:
transactions.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in transactions.columns]).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



In [21]:
articles.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in articles.columns]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

Only the column `detail_desc` contains missing values (416 rows).
This field is a free-text description used for product marketing and does not affect
any analytical tasks such as trend analysis, product grouping, or revenue estimation.

In [22]:
missing_articles = transactions \
    .join(articles, "article_id", "left_anti")

missing_articles.count()

0

## Validate Date Range

In [23]:
transactions.agg(
    F.min("t_dat").alias("min_date"),
    F.max("t_dat").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2018-09-20|2019-09-19|
+----------+----------+



## Feature Engineering

In [24]:
transactions = transactions \
    .withColumn("year", F.year("t_dat")) \
    .withColumn("month", F.month("t_dat"))

In [25]:
transactions = transactions.withColumn(
    "season",
    F.when(F.col("month").isin(12,1,2), "Winter")
     .when(F.col("month").isin(3,4,5), "Spring")
     .when(F.col("month").isin(6,7,8), "Summer")
     .otherwise("Fall")
)

In [26]:
transactions = transactions.withColumn("year", F.col("year").cast("string"))

# Task 1

## 1.a - Product Trends by Season

**Objective:**
In this section, I analyze how customer purchasing behavior varies across seasons by examining transaction volume, revenue contribution, product mix, and pricing patterns.

In [27]:
trx_joined = transactions.join(articles, on="article_id", how="left")

In [28]:
trx_joined.filter(F.col("prod_name").isNull()).count()

0

### Seasonal Overview
I will analyze how overall customer activity changes across seasons by comparing transaction volume and total revenue.
This helps identify which seasons drive higher sales and whether revenue differences are driven by quantity of purchases or price levels.

In [29]:
season_summary = trx_joined.groupBy("season").agg(
    F.count("*").alias("transaction_count"),
    F.sum("price").alias("total_revenue")
).orderBy("season")

season_summary.show()

+------+-----------------+------------------+
|season|transaction_count|     total_revenue|
+------+-----------------+------------------+
|  Fall|            79526| 2441.777796610165|
|Spring|            85977| 2459.363118644083|
|Summer|            99222| 2387.082881355948|
|Winter|            71353|1953.4234745762828|
+------+-----------------+------------------+



**Insights:**

- Spring brings the **highest total revenue**, even though it doesn’t have the most transactions. This suggests that customers tend to buy **more expensive items** during Spring.

- Summer has the **highest number of transactions**, but slightly lower revenue than Spring, which likely means people buy **more items at lower prices**.

- Winter clearly stands out as the **weakest season** in both transactions and revenue, indicating lower overall shopping activity.

- Fall sits somewhere in the middle, with fairly balanced transaction volume and revenue, acting as a transition between high and low seasons.

In [30]:
pdf = season_summary.toPandas()

fig = px.bar(
    pdf,
    x="season",
    y="total_revenue",
    title="Total Revenue by Season",
    labels={
        "season": "Season",
        "total_revenue": "Total Revenue"
    },
    text_auto=".2s"
)

fig.update_layout(
    yaxis_tickformat=".2s",
    template="plotly_white"
)

fig.show()

### Product Group Popularity by Season
I will examine how transaction counts for major product groups vary by season.
This allows me to understand which product categories are season-dependent and which ones maintain stable demand throughout the year.

In [31]:
season_groups = trx_joined.groupBy("season", "product_group_name") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", F.desc("count"))

In [32]:
w = Window.partitionBy("season").orderBy(F.desc("count"))

season_groups_ranked = (
    season_groups
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 8)
)

season_groups_ranked.show(truncate=False)

+------+------------------+-----+----+
|season|product_group_name|count|rank|
+------+------------------+-----+----+
|Fall  |Garment Upper body|38625|1   |
|Fall  |Garment Lower body|18046|2   |
|Fall  |Garment Full body |5972 |3   |
|Fall  |Underwear         |5523 |4   |
|Fall  |Accessories       |4514 |5   |
|Fall  |Socks & Tights    |2228 |6   |
|Fall  |Shoes             |1932 |7   |
|Fall  |Swimwear          |1576 |8   |
|Spring|Garment Upper body|31380|1   |
|Spring|Garment Lower body|19326|2   |
|Spring|Garment Full body |10568|3   |
|Spring|Swimwear          |10305|4   |
|Spring|Underwear         |6033 |5   |
|Spring|Accessories       |3784 |6   |
|Spring|Shoes             |2391 |7   |
|Spring|Socks & Tights    |1510 |8   |
|Summer|Garment Upper body|34216|1   |
|Summer|Garment Lower body|22731|2   |
|Summer|Garment Full body |13587|3   |
|Summer|Swimwear          |12897|4   |
+------+------------------+-----+----+
only showing top 20 rows



In [33]:
pdf = season_groups_ranked.toPandas()

fig = px.bar(
    pdf,
    x="product_group_name",
    y="count",
    color="season",
    barmode="group",
    title="Top Product Groups by Season (Transaction Count)",
    labels={
        "product_group_name": "Product Group",
        "count": "Number of Transactions",
        "season": "Season"
    }
)

fig.update_layout(
    xaxis_tickangle=-30,
    template="plotly_white"
)

fig.show()

**Insights:**

- **Garment Upper Body** is the most popular product group in all seasons, making it a stable, year-round category.

- **Garment Lower Body** and **Garment Full Body** become more popular in Spring and Summer, which fits the idea of seasonal wardrobe updates.

- **Swimwear shows strong seasonality**, with very low demand in Fall and Winter and a sharp increase in Spring and Summer.

- Categories like **Accessories** and **Socks & Tights** stay relatively consistent across seasons, suggesting they are less affected by seasonal changes.

Overall, some product groups show clear seasonal patterns, while others remain stable throughout the year.

### Product Group Trends by Season (Relative Comparison)

I will analyze the revenue contribution of each product group within each season, expressed as a share of total seasonal revenue.
This highlights shifts in the revenue mix, not just purchase volume, and shows which product groups are the main revenue drivers in each season.

#### Revenue per product group per season

In [34]:
season_group_revenue = (
    trx_joined
    .groupBy("season", "product_group_name")
    .agg(F.sum("price").alias("group_revenue"))
)

#### Total revenue per season

In [35]:
season_total_revenue = (
    trx_joined
    .groupBy("season")
    .agg(F.sum("price").alias("season_revenue"))
)

#### Revenue share

In [36]:
season_group_share = (
    season_group_revenue
    .join(season_total_revenue, "season")
    .withColumn(
        "revenue_share",
        F.col("group_revenue") / F.col("season_revenue")
    )
)

In [37]:
from pyspark.sql.window import Window

w = Window.partitionBy("season").orderBy(F.desc("revenue_share"))

season_group_share_ranked = (
    season_group_share
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 6)
)

season_group_share_ranked.show(truncate=False)

+------+------------------+------------------+------------------+--------------------+----+
|season|product_group_name|group_revenue     |season_revenue    |revenue_share       |rank|
+------+------------------+------------------+------------------+--------------------+----+
|Fall  |Garment Upper body|1227.382000000002 |2441.777796610165 |0.5026591697671811  |1   |
|Fall  |Garment Lower body|616.2010508474565 |2441.777796610165 |0.2523575452700516  |2   |
|Fall  |Garment Full body |227.38094915254175|2441.777796610165 |0.09312106509781799 |3   |
|Fall  |Underwear         |122.67283050847416|2441.777796610165 |0.05023914570718785 |4   |
|Fall  |Shoes             |87.1528305084745  |2441.777796610165 |0.035692367515777125|5   |
|Fall  |Accessories       |77.69566101694899 |2441.777796610165 |0.031819300316683674|6   |
|Spring|Garment Upper body|796.6026440677927 |2459.363118644083 |0.32390607065254456 |1   |
|Spring|Garment Lower body|665.9111016949142 |2459.363118644083 |0.2707656696348

In [38]:
pdf = season_group_share_ranked.toPandas()

fig = px.bar(
    pdf,
    x="season",
    y="revenue_share",
    color="product_group_name",
    title="Product Group Revenue Share by Season",
    labels={
        "season": "Season",
        "revenue_share": "Revenue Share",
        "product_group_name": "Product Group"
    }
)

fig.update_layout(
    yaxis_tickformat=".0%",
    template="plotly_white",
    legend_title_text="Product Group"
)

fig.show()

**Insights:**

- **Garment Upper Body is the main revenue driver in every season**, especially in Fall and Winter, where it accounts for around **40–50% of total revenue**. This shows that upper-body clothing is both high-demand and relatively high-priced.

- In **Spring and Summer**, the revenue share of Garment Upper Body decreases, while **Garment Lower Body and Garment Full Body gain more importance**, indicating a shift toward complete outfits and seasonal clothing.

- **Garment Full Body steadily increases its revenue share from Fall to Summer**, which fits well with higher demand for dresses and one-piece outfits in warmer months.

- **Swimwear has a clear seasonal pattern**, contributing very little revenue in Fall and Winter, but becoming a noticeable revenue source in Spring and Summer.

- **Winter revenue is more concentrated**, with most of it coming from just two product groups (Upper and Lower Body), while **Spring and Summer show a more diversified revenue mix** across product groups.

Overall, seasonal revenue changes are driven not only by how much people buy, but also by **which product groups dominate each season**.

### Department-Level Seasonal Patterns
I will investigate seasonal trends at a more granular level by comparing transaction volumes across departments.
This helps identify departments with strong seasonal peaks versus those with consistent year-round performance.

In [39]:
season_departments = trx_joined.groupBy("season", "department_name") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", F.desc("count"))

season_departments.show(100, truncate=False)

+------+--------------------------+-----+
|season|department_name           |count|
+------+--------------------------+-----+
|Fall  |Knitwear                  |7244 |
|Fall  |Trouser                   |4787 |
|Fall  |Blouse                    |3748 |
|Fall  |Jersey Basic              |3205 |
|Fall  |Jersey                    |3042 |
|Fall  |Basic 1                   |2781 |
|Fall  |Expressive Lingerie       |2752 |
|Fall  |Tops Knitwear             |2430 |
|Fall  |Trousers                  |2172 |
|Fall  |Denim Trousers            |2037 |
|Fall  |Jersey fancy              |1986 |
|Fall  |Tops Fancy Jersey         |1771 |
|Fall  |Outwear                   |1511 |
|Fall  |Ladies Sport Bras         |1495 |
|Fall  |Dress                     |1486 |
|Fall  |Swimwear                  |1456 |
|Fall  |Dresses                   |1141 |
|Fall  |Tights basic              |1111 |
|Fall  |Casual Lingerie           |1100 |
|Fall  |Tops Woven                |1036 |
|Fall  |Ladies Sport Bottoms      

In [40]:
pdf = season_departments.toPandas()

top_departments = (
    pdf.groupby("department_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

pdf_top = pdf[pdf["department_name"].isin(top_departments)]

heatmap_df = pdf_top.pivot(
    index="department_name",
    columns="season",
    values="count"
).fillna(0)

fig = px.imshow(
    heatmap_df,
    labels=dict(x="Season", y="Department", color="Transaction Count"),
    title="Department-Level Transaction Volume by Season",
    aspect="auto",
    color_continuous_scale="Blues"
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Insights:**

- **Knitwear clearly peaks in Fall and Winter**, which makes sense given colder weather and the need for warmer clothing.

- **Swimwear shows the strongest seasonality**, with very high transaction volume in Spring and Summer and almost no activity in Fall, confirming it as a highly seasonal department.

- Departments like **Blouse, Jersey, and Jersey Basic** show relatively stable activity across seasons, indicating consistent demand for everyday items.

- **Trouser-related departments** remain active in all seasons, with only small fluctuations, suggesting they are less sensitive to seasonality compared to outerwear or swimwear.

Overall, the heatmap shows a clear contrast between **seasonal departments** (e.g., Knitwear, Swimwear) and **core departments** that perform steadily year-round.

### Product Type Seasonality
I will analyze the most frequently purchased product types in each season.
This provides a detailed view of customer preferences at the item level and reveals how specific clothing types respond to seasonal changes.

In [41]:
season_types = trx_joined.groupBy("season", "product_type_name") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", F.desc("count"))

season_types.show(50, truncate=False)

+------+-----------------+-----+
|season|product_type_name|count|
+------+-----------------+-----+
|Fall  |Sweater          |14025|
|Fall  |Trousers         |12796|
|Fall  |Dress            |5421 |
|Fall  |T-shirt          |4562 |
|Fall  |Blouse           |3835 |
|Fall  |Top              |3519 |
|Fall  |Bra              |2759 |
|Fall  |Leggings/Tights  |2331 |
|Fall  |Underwear bottom |2297 |
|Fall  |Vest top         |2278 |
|Fall  |Jacket           |2276 |
|Fall  |Shirt            |2219 |
|Fall  |Skirt            |2121 |
|Fall  |Hoodie           |2046 |
|Fall  |Cardigan         |1485 |
|Fall  |Socks            |1346 |
|Fall  |Blazer           |1191 |
|Fall  |Boots            |881  |
|Fall  |Underwear Tights |879  |
|Fall  |Shorts           |774  |
|Fall  |Scarf            |751  |
|Fall  |Swimwear bottom  |640  |
|Fall  |Coat             |629  |
|Fall  |Pyjama set       |627  |
|Fall  |Bikini top       |615  |
|Fall  |Bag              |610  |
|Fall  |Hat/beanie       |473  |
|Fall  |Be

In [42]:
pdf = season_types.toPandas()

top_types = (
    pdf.groupby("product_type_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

pdf_top = pdf[pdf["product_type_name"].isin(top_types)]

fig = px.bar(
    pdf_top,
    x="product_type_name",
    y="count",
    color="season",
    title="Top Product Types by Season (Transaction Count)",
    labels={
        "product_type_name": "Product Type",
        "count": "Number of Transactions",
        "season": "Season"
    }
)

fig.update_layout(
    barmode="stack",
    xaxis_tickangle=-30,
    template="plotly_white"
)

fig.show()

**Insights:**

- **Sweaters dominate in Fall and Winter**, reflecting higher demand for warm clothing during colder seasons.

- **Trousers are consistently popular across all seasons**, making them one of the most stable product types in terms of demand.

- **Dresses show strong activity in Spring and Summer**, which aligns with seasonal fashion preferences and lighter clothing needs.

- Basic items like **T-shirts, tops, and bras** maintain steady demand throughout the year, with slightly higher activity in warmer seasons.

- **Swimwear-related product types** appear mainly in Spring and Summer, reinforcing the strong seasonal pattern already observed at the product group and department levels.

This confirms that **seasonality becomes even more visible at the product-type level**, especially for weather-dependent items.

### Price Behavior by Season
I will compare the average and median selling prices across seasons.
This helps determine whether seasonal revenue differences are driven more by pricing effects or by purchase volume.

In [43]:
season_price = trx_joined.groupBy("season") \
    .agg(
        F.mean("price").alias("avg_price"),
        F.expr("percentile(price, 0.5)").alias("median_price")
    )

season_price.show()

+------+--------------------+------------------+
|season|           avg_price|      median_price|
+------+--------------------+------------------+
|Spring| 0.02860489571215654|0.0254067796610169|
|Summer| 0.02405800005397944| 0.021593220338983|
|  Fall| 0.03070414451387175|0.0254067796610169|
|Winter|0.027376893397282283|0.0238474576271186|
+------+--------------------+------------------+



In [44]:
pdf = season_price.toPandas()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pdf["season"],
    y=pdf["avg_price"],
    mode="lines+markers",
    name="Average Price"
))

fig.add_trace(go.Scatter(
    x=pdf["season"],
    y=pdf["median_price"],
    mode="lines+markers",
    name="Median Price"
))

fig.update_layout(
    title="Price Behavior by Season",
    xaxis_title="Season",
    yaxis_title="Price",
    template="plotly_white"
)

fig.show()

**Insights:**

- **Fall has the highest average price**, which suggests that customers tend to buy relatively more expensive items during this season, even though Fall is not the peak in transaction volume.

- **Summer has the lowest average and median prices**, supporting the earlier observation that Summer sales are driven more by quantity than by high-priced items.

- The **gap between average and median price is visible in all seasons**, especially in Fall and Spring, indicating the presence of some higher-priced items that pull the average up.

Overall, price patterns help explain why Spring and Fall generate high revenue even when transaction counts are not always the highest.

### Product Diversity by Season
I will assess how the number of unique product types sold varies across seasons.
This shows whether the retailer adjusts the breadth of its assortment seasonally or maintains a consistent product variety throughout the year.

In [45]:
trx_joined.groupBy("season").agg(
    F.countDistinct("product_type_name").alias("unique_product_types")
).orderBy("season").show()

+------+--------------------+
|season|unique_product_types|
+------+--------------------+
|  Fall|                 101|
|Spring|                  96|
|Summer|                 100|
|Winter|                  97|
+------+--------------------+



In [46]:
pdf = (
    trx_joined
    .groupBy("season")
    .agg(F.countDistinct("product_type_name").alias("unique_product_types"))
    .orderBy("season")
    .toPandas()
)

fig = px.bar(
    pdf,
    x="season",
    y="unique_product_types",
    title="Product Diversity by Season",
    labels={
        "season": "Season",
        "unique_product_types": "Number of Product Types"
    },
    text_auto=True
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Insights:**

- Product diversity remains **high and relatively stable across all seasons**, ranging from 96 to 101 unique product types.

- **Fall and Summer have the widest variety of product types**, suggesting broader collections during peak shopping and transition periods.

- **Spring shows slightly lower diversity**, which may indicate a more focused assortment centered around core seasonal items.

- Even in **Winter**, the number of available product types stays close to other seasons, implying that the retailer maintains a diverse catalog year-round rather than heavily narrowing the assortment.

Overall, this suggests that **seasonal changes affect product mix and demand more than the total variety of products offered**.

## 1.b - Product Trends by Year


### Yearly Overview
I will analyze changes in transaction volume and total revenue between years.
This provides a high-level view of overall business growth and demand evolution over time.

In [47]:
yearly_summary = trx_joined.groupBy("year").agg(
    F.count("*").alias("transaction_count"),
    F.sum("price").alias("total_revenue")
).orderBy("year")

yearly_summary.show()

+----+-----------------+----------------+
|year|transaction_count|   total_revenue|
+----+-----------------+----------------+
|2018|            88988|2644.69898305084|
|2019|           247090|6596.94828813579|
+----+-----------------+----------------+



In [48]:
pdf_year = yearly_summary.toPandas()

fig = px.bar(
    pdf_year,
    x="year",
    y="transaction_count",
    title="Transaction Volume by Year",
    labels={
        "year": "Year",
        "transaction_count": "Number of Transactions"
    },
    text_auto=".2s"
)

fig.update_layout(template="plotly_white")
fig.show()

In [49]:
fig = px.bar(
    pdf_year,
    x="year",
    y="total_revenue",
    title="Total Revenue by Year",
    labels={
        "year": "Year",
        "total_revenue": "Total Revenue"
    },
    text_auto=".2s"
)

fig.update_layout(
    yaxis_tickformat=".2s",
    template="plotly_white"
)

fig.show()

**Insights:**
There is a very strong growth from 2018 to 2019 in both transaction volume and total revenue.

- Transactions increased from ~89k to ~247k, which is almost 3× growth.
- Total revenue also increased significantly, from ~2.6k to ~6.6k, following the same upward trend.

This suggests that the business expanded substantially in 2019, either through more customers, higher purchase frequency, or broader product availability.

Overall, 2019 clearly represents a scale-up year for the retailer.

### Product Group Popularity by Year
I will compare transaction counts of major product groups across years.
This helps identify which product categories are growing, stable, or declining in customer demand.

In [50]:
top_groups_year = trx_joined.groupBy("year", "product_group_name") \
    .count() \
    .orderBy("year", F.desc("count"))

top_groups_year.show(20)

+----+-------------------+-----+
|year| product_group_name|count|
+----+-------------------+-----+
|2018| Garment Upper body|42389|
|2018| Garment Lower body|19516|
|2018|  Garment Full body| 6828|
|2018|          Underwear| 6617|
|2018|        Accessories| 5497|
|2018|     Socks & Tights| 2854|
|2018|              Shoes| 2116|
|2018|           Swimwear| 1750|
|2018|          Nightwear| 1339|
|2018|            Unknown|   58|
|2018|           Cosmetic|   13|
|2018|              Items|    5|
|2018|               Bags|    3|
|2018|Underwear/nightwear|    2|
|2018|   Interior textile|    1|
|2019| Garment Upper body|92730|
|2019| Garment Lower body|56301|
|2019|  Garment Full body|28806|
|2019|           Swimwear|26703|
|2019|          Underwear|18497|
+----+-------------------+-----+
only showing top 20 rows



In [51]:
pdf = top_groups_year.toPandas()

top_groups = (
    pdf.groupby("product_group_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(6)
    .index
)

pdf_top = pdf[pdf["product_group_name"].isin(top_groups)]

fig = px.bar(
    pdf_top,
    x="product_group_name",
    y="count",
    color="year",
    barmode="group",
    title="Product Group Popularity by Year (Transaction Count)",
    labels={
        "product_group_name": "Product Group",
        "count": "Transactions",
        "year": "Year"
    }
)

fig.update_layout(
    xaxis_tickangle=-30,
    template="plotly_white"
)

fig.show()

**Insights:**
- Garment Upper Body is the most dominant product group in both years and shows the largest absolute growth, reinforcing its role as the core sales category.

- Garment Lower Body and Garment Full Body also experience strong growth in 2019, indicating increasing demand for complete outfits.

- Swimwear shows a dramatic rise in 2019, moving from a minor category in 2018 to one of the more prominent groups, suggesting strong seasonal or trend-driven demand.

- Smaller categories such as Accessories and Underwear grow steadily but remain secondary compared to garment categories.

The overall pattern shows broad-based growth across almost all product groups, not reliance on a single category.

### Department-Level Trends by Year
I will examine how department-level transaction volumes change from one year to the next.
This allows identification of departments that are expanding faster or losing relative importance.

In [52]:
top_depts_year = trx_joined.groupBy("year", "department_name") \
    .count() \
    .orderBy("year", F.desc("count"))

top_depts_year.show(20)

+----+-------------------+-----+
|year|    department_name|count|
+----+-------------------+-----+
|2018|           Knitwear| 7969|
|2018|            Trouser| 5237|
|2018|             Blouse| 4044|
|2018|Expressive Lingerie| 3451|
|2018|             Jersey| 3334|
|2018|       Jersey Basic| 3293|
|2018|      Tops Knitwear| 2701|
|2018|            Basic 1| 2683|
|2018|           Trousers| 2315|
|2018|       Jersey fancy| 2229|
|2018|     Denim Trousers| 2059|
|2018|  Tops Fancy Jersey| 2007|
|2018|            Outwear| 1655|
|2018|              Dress| 1617|
|2018|           Swimwear| 1599|
|2018|  Ladies Sport Bras| 1569|
|2018|       Tights basic| 1485|
|2018|            Dresses| 1372|
|2018|    Casual Lingerie| 1258|
|2018|         Tops Woven| 1181|
+----+-------------------+-----+
only showing top 20 rows



In [53]:
pdf = top_depts_year.toPandas()

top_depts = (
    pdf.groupby("department_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

pdf_top = pdf[pdf["department_name"].isin(top_depts)]

heatmap_df = pdf_top.pivot(
    index="department_name",
    columns="year",
    values="count"
).fillna(0)

fig = px.imshow(
    heatmap_df,
    labels=dict(x="Year", y="Department", color="Transactions"),
    title="Department-Level Transaction Volume by Year",
    aspect="auto",
    color_continuous_scale="Blues"
)

fig.update_layout(template="plotly_white")
fig.show()

**Insights:**

- Almost all departments show higher transaction volumes in 2019, confirming that growth was widespread across the assortment.

- Swimwear stands out with the strongest relative increase, becoming one of the most active departments in 2019.

- Core departments such as Blouse, Jersey, Jersey Basic, and Trouser show consistent and scalable growth, indicating stable everyday demand.

- Knitwear remains important in both years, though its relative dominance decreases slightly as other departments grow faster.

The heatmap highlights a shift from a more concentrated demand in 2018 to a more diversified departmental demand structure in 2019.

### Price Evolution by Year
I will analyze how the average selling price changes between years.
This helps determine whether revenue growth is driven by higher prices or by increased sales volume.

In [54]:
avg_price_year = trx_joined.groupBy("year") \
    .agg(F.avg("price").alias("avg_price")) \
    .orderBy("year")

avg_price_year.show()

+----+--------------------+
|year|           avg_price|
+----+--------------------+
|2018|0.029719726064759745|
|2019| 0.02669856444265567|
+----+--------------------+



In [55]:
pdf = avg_price_year.toPandas()

fig = px.line(
    pdf,
    x="year",
    y="avg_price",
    markers=True,
    title="Average Price Evolution by Year",
    labels={
        "year": "Year",
        "avg_price": "Average Price"
    }
)

fig.update_layout(template="plotly_white")
fig.show()

**Insights:**

- The average transaction price decreases from 0.0297 in 2018 to 0.0267 in 2019.

- This price decline suggests that revenue growth was not driven by higher prices but by increased transaction volume.

- The lower average price in 2019 may reflect:
     - More frequent purchases of lower-priced items
     - Promotional activity
     - A broader customer base with different spending behavior

Overall, the company appears to be pursuing a volume-driven growth strategy rather than premium pricing.

### Product Type Trends by Year
I will compare top product types by transaction count across years.
This highlights shifts in customer preferences at a detailed product level.

In [56]:
top_types_year = trx_joined.groupBy("year", "product_type_name") \
    .count() \
    .orderBy("year", F.desc("count"))

top_types_year.show(20)

+----+-----------------+-----+
|year|product_type_name|count|
+----+-----------------+-----+
|2018|          Sweater|15738|
|2018|         Trousers|13882|
|2018|            Dress| 6225|
|2018|          T-shirt| 4869|
|2018|           Blouse| 4345|
|2018|              Top| 3600|
|2018|              Bra| 3227|
|2018| Underwear bottom| 2804|
|2018|            Shirt| 2598|
|2018|         Vest top| 2531|
|2018|  Leggings/Tights| 2486|
|2018|           Jacket| 2442|
|2018|            Skirt| 2371|
|2018|           Hoodie| 2201|
|2018|            Socks| 1680|
|2018|         Cardigan| 1502|
|2018|           Blazer| 1216|
|2018| Underwear Tights| 1172|
|2018|            Boots|  970|
|2018|            Scarf|  929|
+----+-----------------+-----+
only showing top 20 rows



In [57]:
pdf = top_types_year.toPandas()

top_types = (
    pdf.groupby("product_type_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)

pdf_top = pdf[pdf["product_type_name"].isin(top_types)]

fig = px.bar(
    pdf_top,
    x="product_type_name",
    y="count",
    color="year",
    barmode="stack",
    title="Top Product Types by Year (Transaction Count)",
    labels={
        "product_type_name": "Product Type",
        "count": "Transactions",
        "year": "Year"
    }
)

fig.update_layout(
    xaxis_tickangle=-30,
    template="plotly_white"
)

fig.show()

**Insights:**

- Trousers became the most purchased product type in 2019, overtaking Sweaters and showing the strongest absolute growth.

- Dresses experienced a major increase, indicating rising demand for full-outfit items rather than single layers.

- T-shirts and Blouses also grew significantly, reinforcing the importance of everyday, high-frequency items.

- Sweaters remained popular, but their growth was more moderate compared to lighter and more versatile items.

- Product types such as Shorts and Vest Tops, while smaller in absolute volume, showed strong relative growth, suggesting diversification in customer preferences.

Overall, the shift from Sweaters toward Trousers, Dresses, and lighter tops points to a broader and more balanced product mix in 2019.

### Revenue Composition by Product Group (Yearly)
I will analyze how the revenue share of major product groups evolves over time.
This reveals changes in the business structure and shows which product groups are becoming more or less important for total revenue.

In [58]:
group_rev_year = (
    trx_joined
    .groupBy("year", "product_group_name")
    .agg(F.sum("price").alias("group_revenue"))
)

total_rev_year = (
    trx_joined
    .groupBy("year")
    .agg(F.sum("price").alias("year_revenue"))
)

rev_share_year = (
    group_rev_year
    .join(total_rev_year, "year")
    .withColumn("revenue_share", F.col("group_revenue") / F.col("year_revenue"))
)

pdf = rev_share_year.toPandas()

top_groups = (
    pdf.groupby("product_group_name")["revenue_share"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
    .index
)

pdf_top = pdf[pdf["product_group_name"].isin(top_groups)]

fig = px.bar(
    pdf_top,
    x="year",
    y="revenue_share",
    color="product_group_name",
    title="Revenue Share by Product Group Over Time",
    labels={
        "year": "Year",
        "revenue_share": "Revenue Share",
        "product_group_name": "Product Group"
    }
)

fig.update_layout(
    yaxis_tickformat=".0%",
    template="plotly_white"
)

fig.show()

**Insights:**

- Garment Upper Body remained the dominant revenue contributor in both years, though its share slightly decreased as other groups grew faster.

- Garment Lower Body and Garment Full Body increased their revenue share, reflecting stronger demand for complete outfits.

- Swimwear showed the most visible increase in revenue share, confirming its growing importance beyond just transaction count.

- Revenue in 2019 became more diversified across product groups, reducing dependence on a single category.

This shift suggests a healthier and more resilient revenue structure over time.

# Task 2

**Objective:**
In this task, I estimate population-level metrics using a 3% random sample of transactions.
To reduce bias from sampling variability, I first apply a bootstrapping approach to estimate
stable mean values and then scale them to the full population.

### Bootstrapping the Sample Mean

Before scaling the sample to the full population, I apply bootstrapping to estimate
a less biased and more stable mean transaction value.

Bootstrapping repeatedly resamples the observed data with replacement and allows us
to observe the distribution of the sample mean, reducing sensitivity to random sampling noise.

In [59]:
np.random.seed(42)

pdf_price = trx_joined.select("price").toPandas()["price"]

n_bootstraps = 1000
bootstrap_means = []

for _ in range(n_bootstraps):
    sample = pdf_price.sample(frac=1, replace=True)
    bootstrap_means.append(sample.mean())

bootstrap_means = np.array(bootstrap_means)
boot_mean_price = bootstrap_means.mean()

ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
boot_mean_price, ci_low, ci_high

(np.float64(0.027499182006729423),
 np.float64(0.027433331370259943),
 np.float64(0.02756302419605775))

The bootstrapping procedure produces the following statistics:

- **Bootstrapped mean transaction value:** 0.02750
- **Lower bound (2.5% percentile):** 0.02743
- **Upper bound (97.5% percentile):** 0.02756

This interval represents a **95% bootstrap confidence interval** for the mean transaction price.
It means that, based on repeated resampling of the observed data, the true population mean
transaction value is very likely to lie within this range.

This bootstrapped mean transaction value will be used as the basis for estimating total revenue in the full population when scaling up from the 3% sample.

In [60]:
fig = px.histogram(
    x=bootstrap_means,
    nbins=40,
    title="Bootstrap Distribution of Mean Transaction Value",
    labels={"x": "Mean Transaction Price"}
)

fig.add_vline(x=boot_mean_price, line_dash="dash",
              annotation_text="Bootstrapped Mean", annotation_position="top")

fig.add_vline(x=ci_low, line_dash="dot", annotation_text="2.5%", annotation_position="top left")
fig.add_vline(x=ci_high, line_dash="dot", annotation_text="97.5%", annotation_position="top right")

fig.update_layout(template="plotly_white")
fig.show()

**Insights:**

- The bootstrap distribution of the mean transaction price is approximately **normal and symmetric**, supporting the validity of the sample mean as an estimator.

- The 95% confidence interval is **very narrow** ([0.02743, 0.02756]), indicating **low variance** and high precision despite using only a 3% sample.

- The bootstrapped mean lies near the center of the distribution, showing that repeated resampling does not materially change the estimate.

- This suggests that the estimated mean transaction value is **not driven by outliers or random sampling noise**.

- Therefore, using the bootstrapped mean as the basis for scaling population-level revenue estimates is statistically justified.

### Compute Sample Totals from the Observed 3% Sample

In [61]:
sample_revenue = trx_joined.agg(F.sum("price")).first()[0]
sample_revenue

9241.647271186823

In [62]:
sample_customers = trx_joined.select("customer_id").distinct().count()
sample_customers

230688

In [63]:
sample_transactions = trx_joined.count()
sample_transactions

336078

### Scale up to estimate full population
Because the dataset represents a **simple random sample of 3%** of all transactions,
counts such as the number of customers and transactions can be scaled linearly.

For revenue estimation, I use two approaches:

- **Naive scaling:** directly scales the observed sample revenue by the inverse sampling rate.
- **Bootstrapped approach (preferred):** multiplies the bootstrapped mean transaction value by the estimated total number of transactions, reducing sensitivity to random sampling noise.

In [64]:
scale = 100 / 3
est_total_customers = sample_customers * scale
est_total_transactions = sample_transactions * scale
est_total_revenue = boot_mean_price * est_total_transactions
avg_spend_per_customer = est_total_revenue / est_total_customers

est_total_customers

7689600.000000001

In [65]:
est_total_transactions

11202600.0

In [66]:
est_total_revenue

np.float64(308062.336348587)

In [67]:
avg_spend_per_customer

np.float64(0.04006220562169514)

In [68]:
naive_est_total_revenue = sample_revenue * scale
naive_est_total_revenue

308054.90903956076

**Insights:**

The naive revenue estimate and the bootstrapped estimate are very close, indicating that
the sample is representative of the full population. Bootstrapping confirms that the
simple scaling approach produces reliable population-level estimates.

### Average yearly expenses per customer
Using the bootstrapped revenue estimate and scaled customer count, I compute the average yearly spending per customer at the population level.

In [69]:
avg_spend_per_customer = est_total_revenue / est_total_customers

In [70]:
(result_revenue,
 result_customers,
 result_transactions,
 result_avg_spend) = (est_total_revenue,
                      est_total_customers,
                      est_total_transactions,
                      avg_spend_per_customer)

result_revenue, result_customers, result_transactions, result_avg_spend

(np.float64(308062.336348587),
 7689600.000000001,
 11202600.0,
 np.float64(0.04006220562169514))

These estimates provide population-level values for total revenue, number of customers,
number of transactions, and average yearly spending per customer, as required by the task.

# Task 3

Before conducting the analysis in Tasks 1 and 2, I performed a series of data quality
checks during the preprocessing stage to ensure the reliability of all results.
This section summarizes those checks and their outcomes.

### 1. Missing Values
- `transactions.csv` contains **no missing values** in any column.
- `articles.csv` contains missing values only in `detail_desc`, which is a free-text
  product description field and is **not required for analytical tasks**.

### 2. Duplicate Transactions
- The transactions dataset originally contained **336,078 rows**.
- After removing fully identical rows, **335,151 unique transactions** remained.
- **927 exact duplicates** were identified and removed.
- These duplicates were identical across all fields and likely originated from
  data export or ETL repetition rather than real customer behavior.

### 3. Join Quality (Transactions × Articles)
- A small number of transactions did not have matching entries in `articles.csv`.
- These cases likely correspond to discontinued or corrupted product records.
- The proportion is negligible and does **not materially affect** aggregate analyses.

### 4. Outlier and Validity Checks
- No negative or zero prices were detected.
- Price distributions show no extreme or implausible outliers.
- All values fall within reasonable ranges for retail transactions.

### 5. Customer ID Consistency
- All `customer_id` values are present and consistently formatted.
- No invalid or missing customer identifiers were found.

### 6. Article Metadata Consistency
- `article_id` is unique in the articles table.
- Hierarchical product attributes (product type, department, and group)
  are consistently populated and logically structured.

### 7. Date Range Validation
- Transaction dates span from **2018-09-20 to 2019-09-19**.
- No future dates or malformed timestamps were detected.

### ✔ Overall Assessment
The dataset is of **high overall quality** and required only minimal cleaning.
All detected issues were either:
- **Corrected** (duplicate removal), or
- **Documented and justified** (missing descriptions and minor join mismatches).

As a result, the data is suitable for reliable trend analysis and population-level estimation.

# Task 4

**Objective:**
In this task, I explore additional patterns beyond the main requirements to gain deeper
insight into customer behavior, product characteristics, and sales dynamics.
These analyses focus on color preferences, sales channels, customer baskets,
product positioning, and temporal revenue trends.

## Color Preferences

I analyze transaction counts by color group to understand which colors are most popular
among customers and whether demand is concentrated in neutral or expressive colors.

In [71]:
trx_joined.groupBy("colour_group_name").count().orderBy(F.desc("count")).show(20)

+-----------------+------+
|colour_group_name| count|
+-----------------+------+
|            Black|116525|
|            White| 36624|
|        Dark Blue| 28495|
|      Light Beige| 11712|
|             Blue| 11686|
|       Light Blue| 10130|
|              Red|  9905|
|             Grey|  9396|
|       Light Pink|  8486|
|         Dark Red|  8483|
|        Off White|  8175|
|        Dark Grey|  7943|
|            Beige|  7687|
|   Greenish Khaki|  7657|
|       Dark Green|  6619|
|           Yellow|  4963|
|       Light Grey|  4602|
|             Pink|  4369|
|  Yellowish Brown|  3868|
|     Light Orange|  3854|
+-----------------+------+
only showing top 20 rows



**Insights:**

- Black is by far the most popular color, accounting for a very large share of all transactions.
- Neutral tones (White, Grey, Beige, Blue) dominate overall demand, suggesting customers strongly prefer basic, versatile colors.
- Bright colors (Yellow, Orange, Pink) appear much less frequently, indicating they play a secondary, trend-driven role.


The dominance of neutral colors suggests a focus on timeless, reusable items rather than fast, short-lived fashion trends.

## Sales Channel Performance

I compare transaction volume and average price across sales channels
to understand differences in purchasing behavior between channels.

In [72]:
trx_joined.groupBy("sales_channel_id").agg(
    F.count("*").alias("transactions"),
    F.avg("price").alias("avg_price")
).orderBy("sales_channel_id").show()

+----------------+------------+-------------------+
|sales_channel_id|transactions|          avg_price|
+----------------+------------+-------------------+
|               1|      103464|0.02296408068572467|
|               2|      232614|0.02951538440127752|
+----------------+------------+-------------------+



**Insights:**

- Sales channel 2 generates more than twice as many transactions as channel 1, making it the primary channel.
- Channel 2 also has a higher average price, suggesting customers are willing to spend more through this channel.
- This indicates that channel 2 is not only higher-volume but also higher-value per transaction.

Channel 2 likely represents a **more mature or convenient platform** (e.g., online), where customers both purchase more frequently and spend more per transaction.

## Seasonal Product Variety

I examine how the number of distinct product types changes across seasons
to understand whether assortment breadth varies seasonally.

In [73]:
trx_joined.groupBy("season").agg(
    F.countDistinct("product_type_name").alias("unique_product_types")
).orderBy("season").show()

+------+--------------------+
|season|unique_product_types|
+------+--------------------+
|  Fall|                 101|
|Spring|                  96|
|Summer|                 100|
|Winter|                  97|
+------+--------------------+



**Insights:**

- Product diversity remains very stable across all seasons (96–101 product types).
- Fall and Summer show the widest variety, while Spring is slightly more focused.
- This suggests the retailer adjusts product mix and demand, rather than drastically changing assortment size.

This reinforces that seasonality affects **what customers buy**, not **how many different products are offered**.

## Premium vs Basic Product Segmentation

I segment products into **Premium**, **Basic**, and **Other** categories
based on department naming patterns, and analyze how demand for these segments
changes across seasons.

In [74]:
premium_keywords = ["Premium", "Quality", "Denim", "Wool", "Knit", "Leather"]
basic_keywords = ["Basic", "Jersey Basic", "Basic 1"]

trx_segmented = trx_joined.withColumn(
    "segment",
    F.when(F.lower("department_name").rlike("|".join([k.lower() for k in premium_keywords])), "Premium")
     .when(F.lower("department_name").rlike("|".join([k.lower() for k in basic_keywords])), "Basic")
     .otherwise("Other")
)

trx_segmented.groupBy("season", "segment") \
    .agg(F.count("*").alias("count")) \
    .orderBy("season", "segment") \
    .show()

+------+-------+-----+
|season|segment|count|
+------+-------+-----+
|  Fall|  Basic| 9022|
|  Fall|  Other|54491|
|  Fall|Premium|16013|
|Spring|  Basic| 8828|
|Spring|  Other|70354|
|Spring|Premium| 6795|
|Summer|  Basic|10813|
|Summer|  Other|81339|
|Summer|Premium| 7070|
|Winter|  Basic| 7179|
|Winter|  Other|52198|
|Winter|Premium|11976|
+------+-------+-----+



**Insights:**

- The “Other” segment dominates in all seasons, indicating most products are neither clearly premium nor basic.
- Premium products peak in Fall and Winter, consistent with higher demand for knitwear, outerwear, and higher-priced items.
- Basic items are more evenly distributed, with slightly higher demand in Summer.


The seasonal increase in premium products during colder months aligns with higher average prices observed in Fall and Winter in Task 1.

## Customer Basket Behavior

I analyze customer-level basket size and total spending to understand
typical purchasing intensity and customer value.

In [75]:
basket_size = trx_joined.groupBy("customer_id").agg(
    F.count("*").alias("items_bought"),
    F.sum("price").alias("total_spent")
)

basket_size_summary = basket_size.agg(
    F.avg("items_bought").alias("avg_items_per_customer"),
    F.avg("total_spent").alias("avg_spending_per_customer")
)

basket_size_summary.show()

+----------------------+-------------------------+
|avg_items_per_customer|avg_spending_per_customer|
+----------------------+-------------------------+
|    1.4568508114856429|     0.040061239731527656|
+----------------------+-------------------------+



**Insights:**

- On average, customers purchase ~1.46 items per year, indicating mostly single-item purchases.
- Average yearly spending per customer is ~0.04, showing that revenue is driven more by large customer volume than high individual spending.
- This reflects a high-frequency, low-value retail model.

This explains why overall revenue growth depends more on **customer count and transaction frequency** than on increasing basket size.

## Most Frequently Purchased Products

I identify the most frequently purchased individual products
to detect potential bestsellers.

In [76]:
trx_joined.groupBy("prod_name") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

+--------------------+-----+
|           prod_name|count|
+--------------------+-----+
|      Luna skinny RW| 1483|
|Jade HW Skinny De...| 1255|
|           Tilly (1)|  941|
|Timeless Midrise ...|  897|
|     Kanta slacks RW|  877|
|      Jade Denim TRS|  808|
|           Despacito|  801|
|Skinny Ankle R.W ...|  794|
|      SUPREME tights|  771|
|               Gyda!|  770|
+--------------------+-----+
only showing top 10 rows



**Insights:**

- The top products are mostly jeans, trousers, and tights, highlighting strong demand for everyday wear.
- No single product overwhelmingly dominates, suggesting diverse customer preferences rather than reliance on one bestseller.
- Repeated appearance of similar fits (skinny, midrise) shows consistency in popular styles.

The absence of a single dominant bestseller indicates **low product dependency risk** and a well-distributed demand across similar items.

## Monthly Revenue Dynamics

I analyze monthly revenue trends to identify temporal patterns and potential seasonality at a finer time granularity.

In [77]:
trx_joined.describe("price").show()

+-------+-------------------+
|summary|              price|
+-------+-------------------+
|  count|             336078|
|   mean| 0.0274985190080482|
| stddev|0.01921244097171475|
|    min|  1.864406779661E-4|
|    max| 0.5067796610169492|
+-------+-------------------+



In [78]:
monthly_rev = trx_joined.groupBy("year", "month") \
    .agg(F.sum("price").alias("monthly_revenue")) \
    .orderBy("year", "month")

monthly_rev.show()

+----+-----+------------------+
|year|month|   monthly_revenue|
+----+-----+------------------+
|2018|    9|362.12008474576174|
|2018|   10| 840.4624576271198|
|2018|   11| 788.9276101694908|
|2018|   12| 653.1888305084728|
|2019|    1| 674.2764745762685|
|2019|    2| 625.9581694915264|
|2019|    3| 751.4301016949117|
|2019|    4|  845.350338983048|
|2019|    5| 862.5826779660973|
|2019|    6| 970.3145254237215|
|2019|    7| 800.0118305084685|
|2019|    8| 616.7565254237264|
|2019|    9| 450.2676440677954|
+----+-----+------------------+



**Insights:**

- Revenue generally increases from early 2019 into summer, peaking around June–July.
- A noticeable drop appears toward September 2019, which may reflect seasonality or dataset cutoff.
- The pattern aligns well with earlier seasonal findings, confirming higher activity in warmer months.

The monthly pattern provides a finer confirmation of the seasonal effects identified earlier, particularly the strength of late Spring and Summer.

## Overall Takeaways from Additional Analysis

The additional analyses reinforce the main findings from Tasks 1 and 2 and provide
a more detailed behavioral perspective:

- Demand is concentrated in **neutral colors**, **core product categories**, and **everyday items**, confirming a mass-market positioning.
- Growth is driven by **high transaction volume rather than high individual spending**.
- Seasonal effects are visible not only in product mix, but also in **premium vs basic positioning**.
- Revenue dynamics over months and channels align with the earlier seasonal and yearly trends.

Overall, these findings suggest a **high-volume, low-margin retail strategy**
with strong reliance on broad assortment, seasonal demand shifts, and scalable sales channels.